# NBA Game Predictor

Predicts the winner of an NBA game from each team's recent form (last 5 games) and season-to-date averages, including the four factors. Trained with XGBoost on regular-season and playoff games from 2000-01 through the 2023 Finals.

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

import nba_predictor as nba

## Data and features

In [2]:
games = nba.add_features(nba.to_team_games(nba.load_games()))
X, y, meta = nba.build_training_table(games)
X.shape

(29504, 32)

## Train/test split

Split by date so the model never sees future games: train on 2000-2019, test on the 2019-20 season onward.

In [3]:
split = pd.Timestamp("2019-10-01")
train, test = meta["game_date"] < split, meta["game_date"] >= split

model = nba.train_model(X[train], y[train])
proba = model.predict_proba(X[test])[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"Accuracy: {accuracy_score(y[test], pred):.3f}")
print(f"ROC AUC:  {roc_auc_score(y[test], proba):.3f}")
print(f"Baseline (always pick home team): {y[test].mean():.3f}")

Accuracy: 0.623
ROC AUC:  0.662
Baseline (always pick home team): 0.555


In [4]:
pd.DataFrame(
    confusion_matrix(y[test], pred),
    index=["actual away win", "actual home win"],
    columns=["pred away win", "pred home win"],
)

,pred away win,pred home win
actual away win,960,1316
actual home win,611,2224


## Most important features

In [5]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

win_pct_season_avg_diff      0.218502
efg_pct_season_avg_diff      0.073208
win_pct_rolling_diff         0.063753
reb_season_avg_diff          0.028332
fg3_pct_season_avg_diff      0.027910
tov_ratio_season_avg_diff    0.027332
pf_season_avg_diff           0.025300
tov_season_avg_diff          0.025220
blk_season_avg_diff          0.024822
fg_pct_season_avg_diff       0.024334
efg_pct_rolling_diff         0.023977
rest_days_diff               0.023349
ast_season_avg_diff          0.022869
ft_pct_season_avg_diff       0.022062
stl_rolling_diff             0.022055
dtype: float32

## Simulating the 2023 Finals

Retrain on every game, then play the Denver-Miami series 1,000 times.

In [6]:
full_model = nba.train_model(X, y)

results = [nba.simulate_series(full_model, games, "DEN", "MIA", "2023-06-01", seed=i)[0] for i in range(1000)]
pd.Series(results).value_counts(normalize=True)

DEN    0.875
MIA    0.125
Name: proportion, dtype: float64